<div style="background: linear-gradient(135deg, #198754 0%, #20c997 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">🎨 04 — Decorators</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 00 · Bloco 02 · Python Intermediário</p>
</div>

## 🎯 Objetivos deste notebook

Decorators são "embalagens" para funções. Eles permitem modificar o comportamento de uma função sem alterar seu código original.
Neste notebook você vai aprender a:
- Entender o fluxo de um decorator.
- Criar decorators reais muito úteis em Data Science: Timer, Logger e Retry.
- Usar o `@functools.wraps`.

---

## 1️⃣ O Conceito de Wrapper (Envoltório)

Um decorator é apenas uma função que recebe outra função como argumento, a envolve (wrap) com algum código extra antes/depois da execução, e retorna essa nova função agrupada.

In [ ]:
def decorador_simples(funcao_original):
    def wrapper():
        print("🔔 [Antes] Preparando a execução...")
        funcao_original()
        print("🔔 [Depois] Execução finalizada.")
    return wrapper

def ola_mundo():
    print("Olá, mundo!")

# Aplicando o decorator manualmente:
ola_decorado = decorador_simples(ola_mundo)
ola_decorado()

## 2️⃣ A Sintaxe Mágica (`@`)

O Python provê o símbolo `@` (Syntactic Sugar) para não precisarmos fazer a atribuição manual que fizemos acima.

In [ ]:
@decorador_simples
def treinar_modelo():
    print("Treinando 10 épocas...")

treinar_modelo()

## 3️⃣ O Decorator Essencial em Data Science: `@timer`

Vamos construir um decorator que nos dirá quanto tempo uma função demorou para executar (tempo de inferência, tempo de query em banco, etc.).

In [ ]:
import time
import functools

def timer(func):
    @functools.wraps(func)  # <- Importante para não perder as docstrings da func original!
    def wrapper(*args, **kwargs):  # Aceita qualquer argumento
        inicio = time.time()
        resultado = func(*args, **kwargs)  # Roda a função de verdade
        fim = time.time()
        print(f"⏱️ A função '{func.__name__}' executou em {fim - inicio:.4f}s")
        return resultado
    return wrapper

@timer
def simulacao_pesada(tamanho):
    """Faz uma soma gigante de quadrados."""
    return sum(i * i for i in range(tamanho))

print("Soma total:", simulacao_pesada(5_000_000))

## 4️⃣ Decorators com Argumentos: `@retry`

E se a função de baixar um dataset na internet der erro de rede? Podemos criar um decorator `@retry(vezes)` que tentará executar a função `X` vezes antes de falhar definitivamente.

In [ ]:
def retry(max_tentativas):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for tentativa in range(1, max_tentativas + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"❌ Falhou (tentativa {tentativa}/{max_tentativas}): {e}")
                    if tentativa == max_tentativas:
                        raise
                    time.sleep(1) # Aguarda 1s antes da próxima
        return wrapper
    return decorator

@retry(max_tentativas=3)
def requisitar_api():
    import random
    if random.random() < 0.7:  # 70% de chance de dar erro
        raise ConnectionError("A rede caiu!")
    return "Dados recebidos com sucesso!"

print("\nIniciando requisição...")
try:
    print(requisitar_api())
except:
    print("Desistimos de tentar.")

In [ ]:
# ✏️ EXERCÍCIO: Crie um decorator @logger que escreva o nome da função e os argumentos passados para ela toda vez que for chamada.

# Seu código aqui

# @logger
# def somar(a, b):
#     return a + b

# somar(5, 7)  # Esperado: [LOG] Chamando a função 'somar' com args=(5, 7) e kwargs={}

---
## ✅ Checkpoint

- [ ] Entendo como a sintaxe `@nome_decorator` funciona nos bastidores.
- [ ] Lembro de usar `*args` e `**kwargs` dentro do wrapper para suportar qualquer função.
- [ ] Entendo por que o `@functools.wraps` é considerado uma boa prática.

➡️ **Próximo:** `05_type_hints.ipynb`